<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase48.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 48 — frozen v15 bootstrap-only replay and Arm C executability decision

Phase 47 identified a reproducibility blocker in the authoritative frozen TFL v15 runner: the notebook contains a Google Drive mount/bootstrap dependency even though all imported Python modules are otherwise available.

Phase 48 performs the narrowest defensible replay:

- verify the exact Phase 47 evidence chain;
- re-download and verify the exact frozen v15 commit archive;
- verify the authoritative runner byte-for-byte;
- create an **execution copy** in which only the `drive.mount(...)` bootstrap statement is replaced because Drive is already mounted by this parent notebook;
- prove that every other frozen code cell remains byte-identical;
- inventory the runner's external file dependencies against Drive;
- attempt the unchanged v15 method execution;
- determine whether the frozen implementation exposes a generic event-construction API that could accept new neuroimaging evidence without changing v15 rules.

### Comparator rule

A bootstrap wrapper may change environment setup. It may **not** change mapping logic, warning rules, fidelity-state logic, metric definitions, or add MRI-specific semantics.

Even if the native v15 replay succeeds, neuroimaging Arm C is not considered executed unless the frozen implementation itself exposes an unmodified path for the study evidence.


In [1]:
# CODE CELL 1/9 — Drive, constants, helpers
import os, re, csv, json, hashlib, shutil, tempfile, zipfile, time, subprocess, sys, ast
import datetime as dt
from pathlib import Path
from collections import Counter, defaultdict

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='48_frozen_v15_bootstrap_only_replay_and_armC_executability_decision'

EXPECTED={
 'shareable':('ICHI2027_Phase47_SHAREABLE','.json','12037ffac27f2afdef083a3f4f2fa5a14c87d8f5b275ad615b73f699bb7a8a10'),
 'native_audit':('ICHI2027_Phase47_NATIVE_V15_EXECUTION_AUDIT','.csv','4e5a941f8d420f66b961b1c499e3346e8f667fea4dcc03dd40470cbf72255ef4'),
 'cell_inventory':('ICHI2027_Phase47_V15_RUNNER_CELL_INVENTORY','.csv','8339634b3dd153319b31222eaeacb9462ad4ee18824e7246a951ac9424ecc296'),
 'asset_refs':('ICHI2027_Phase47_EXTERNAL_ASSET_REFERENCES','.csv','4e2a0319d80354978637b08ef820a57d03c530485717aab06e3611aa52e5c8c6'),
 'function_inventory':('ICHI2027_Phase47_FUNCTION_INVENTORY','.csv','de31ad10398c06e2e5566a0f731121e686f22ab87b7aa3c4b96bb17bfe510b74'),
 'applicability':('ICHI2027_Phase47_E0_E7_GENERIC_V15_APPLICABILITY','.csv','69aff9c4b7925b3868f557f6a503c1529950db22f993bf136f0d1a5348731060'),
}

REPO='SANGHATI23/ohdsi-fhir-omop-showcase-demo'
TAG='tfl-submission-freeze-v15'
COMMIT='a4b7265adabf096e84b957de068e24a5ab853425'
EXPECTED_ARCHIVE_SHA='e729610f050217a2c62c0536309c7e3ac3508f39992d7ed46b1b70c9bc9478dc'
RUNNER_NAME='FHIRy_pyOMOP_TFL_SubmissionFreezeRunner_v15.ipynb'
EXPECTED_RUNNER_SHA='5416d658e18d9d4f3dfb229996d685bd3dbeeb1d5d9ac910443b8527e9f3ff19'

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader(); w.writerows(rows)

if os.environ.get('PHASE48_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE48_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
    print('Offline Drive-mirror mode:',MYDRIVE)
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir(): ROOTS.append(sd)

hint=os.environ.get('PHASE48_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute(): hp=MYDRIVE/hp
    if not hp.is_dir(): raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase48_'))
SRC=WORK/'release'; SRC.mkdir()
STAGE=WORK/'outputs'; STAGE.mkdir()
OUT_BASE=Path(os.environ.get('PHASE48_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase48')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
print('Phase 48 output:',OUT)


Mounted at /content/drive
Phase 48 output: /content/drive/MyDrive/ICHI2027/Phase48/run_20261001T035922_093456Z


In [2]:
# CODE CELL 2/9 — Verify exact Phase 47 evidence chain

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

def find_by_sha(stem,ext,digest):
    good=[]
    for root in ROOTS:
        for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
            dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
            for fn in files:
                if name_matches(fn,stem,ext):
                    p=Path(here)/fn
                    try:
                        if sha256(p)==digest: good.append(p)
                    except OSError: pass
    if not good:
        raise FileNotFoundError(f'No SHA-approved Drive copy found for {stem}{ext}')
    return sorted(good,key=str)[0]

P={}
for key,(stem,ext,digest) in EXPECTED.items():
    P[key]=find_by_sha(stem,ext,digest)

p47=json.loads(P['shareable'].read_text('utf-8'))
with P['native_audit'].open(newline='',encoding='utf-8-sig') as f:native47=list(csv.DictReader(f))
with P['cell_inventory'].open(newline='',encoding='utf-8-sig') as f:cell47=list(csv.DictReader(f))
with P['asset_refs'].open(newline='',encoding='utf-8-sig') as f:assets47=list(csv.DictReader(f))
with P['function_inventory'].open(newline='',encoding='utf-8-sig') as f:functions47=list(csv.DictReader(f))
with P['applicability'].open(newline='',encoding='utf-8-sig') as f:app47=list(csv.DictReader(f))

assert p47['phase']=='47_frozen_tfl_v15_runner_audit_and_neuroimaging_applicability_preregistration'
assert p47['frozen_commit']==COMMIT
assert p47['authoritative_runner_sha256']==EXPECTED_RUNNER_SHA
assert p47['native_v15_execution_status']=='BLOCKED_INTERACTIVE_OR_DRIVE_DEPENDENCY'
assert p47['missing_runtime_modules']==0
assert p47['interactive_or_drive_dependency_detected'] is True
assert p47['frozen_original_generic_tfl_armC_executed'] is False
assert len(native47)==1 and len(cell47)==17 and len(app47)==8

print('PASS: Phase 47 evidence chain verified.')
print('Native v15 blocker:',p47['native_v15_execution_status'])
print('Missing runtime modules:',p47['missing_runtime_modules'])


PASS: Phase 47 evidence chain verified.
Native v15 blocker: BLOCKED_INTERACTIVE_OR_DRIVE_DEPENDENCY
Missing runtime modules: 0


In [3]:
# CODE CELL 3/9 — Re-download exact frozen commit archive and verify runner
import requests

session=requests.Session()
session.headers.update({'User-Agent':'ICHI2027-neuroimaging-fhir-omop-fidelity-phase48'})

archive=WORK/'v15.zip'
r=session.get(f'https://api.github.com/repos/{REPO}/zipball/{COMMIT}',timeout=120,
              headers={'Accept':'application/vnd.github+json'})
r.raise_for_status()
archive.write_bytes(r.content)
assert sha256(archive)==EXPECTED_ARCHIVE_SHA

with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    z.extractall(SRC)

tops=[p for p in SRC.iterdir() if p.is_dir()]
assert len(tops)==1
ROOT=tops[0]
RUNNER=ROOT/RUNNER_NAME
assert RUNNER.is_file()
assert sha256(RUNNER)==EXPECTED_RUNNER_SHA

print('PASS: exact frozen v15 archive and runner verified.')
print('Commit:',COMMIT)
print('Runner SHA-256:',sha256(RUNNER))


PASS: exact frozen v15 archive and runner verified.
Commit: a4b7265adabf096e84b957de068e24a5ab853425
Runner SHA-256: 5416d658e18d9d4f3dfb229996d685bd3dbeeb1d5d9ac910443b8527e9f3ff19


In [4]:
# CODE CELL 4/9 — Create bootstrap-only execution copy and prove method-cell immutability

runner=json.loads(RUNNER.read_text('utf-8'))
expected_by_index={int(r['cell_index']):r for r in cell47}

patch_rows=[]
mount_cells=[]
patched_calls=0

for idx,c in enumerate(runner.get('cells',[])):
    if c.get('cell_type')!='code': continue
    src=''.join(c.get('source',[])) if isinstance(c.get('source',[]),list) else str(c.get('source',''))
    original_sha=hashlib.sha256(src.encode('utf-8')).hexdigest()

    # Cross-check the authoritative runner against Phase 47's per-cell hashes.
    if idx in expected_by_index:
        assert original_sha==expected_by_index[idx]['source_sha256'],(idx,original_sha,expected_by_index[idx]['source_sha256'])

    new_lines=[]
    replacements=0
    for line in src.splitlines(True):
        if 'drive.mount(' in line:
            indent=line[:len(line)-len(line.lstrip())]
            newline='\n' if line.endswith('\n') else ''
            new_lines.append(indent+"print('Phase48 bootstrap adapter: parent notebook already mounted Drive')"+newline)
            replacements+=1
        else:
            new_lines.append(line)

    patched=''.join(new_lines)
    patched_sha=hashlib.sha256(patched.encode('utf-8')).hexdigest()
    if replacements:
        mount_cells.append(idx)
        patched_calls+=replacements
        c['source']=patched
    else:
        assert patched_sha==original_sha

    patch_rows.append({
      'cell_index':idx,
      'original_source_sha256':original_sha,
      'execution_copy_source_sha256':patched_sha,
      'drive_mount_calls_replaced':replacements,
      'method_source_unchanged':replacements==0
    })

assert len(mount_cells)==1,('Expected exactly one Drive-mount code cell',mount_cells)
assert patched_calls>=1
assert sum(not r['method_source_unchanged'] for r in patch_rows)==1

WRAPPER=WORK/'FHIRy_pyOMOP_TFL_SubmissionFreezeRunner_v15_BOOTSTRAP_WRAPPER.ipynb'
WRAPPER.write_text(json.dumps(runner,indent=1),encoding='utf-8')

print('Bootstrap-only execution copy created.')
print('Patched cell:',mount_cells[0])
print('Drive-mount calls replaced:',patched_calls)
print('All other code cells remain byte-identical.')


Bootstrap-only execution copy created.
Patched cell: 3
Drive-mount calls replaced: 1
All other code cells remain byte-identical.


In [5]:
# CODE CELL 5/9 — Classify external asset references and search Drive without exposing private paths

runner_original=json.loads(RUNNER.read_text('utf-8'))
runner_code='\n'.join(
    ''.join(c.get('source',[])) if isinstance(c.get('source',[]),list) else str(c.get('source',''))
    for c in runner_original.get('cells',[]) if c.get('cell_type')=='code'
)

READ_TOKENS=['read_csv','read_parquet','read_json','pd.read_','open(','zipfile.zipfile','zipfile.ZipFile']
WRITE_TOKENS=['to_csv','to_parquet','to_json','write_text','write_bytes','writestr','.write(']

def classify_ref(ref):
    positions=[m.start() for m in re.finditer(re.escape(ref),runner_code)]
    read=False; write=False
    for pos in positions:
        context=runner_code[max(0,pos-220):pos+len(ref)+220]
        low=context.lower()
        if any(t.lower() in low for t in READ_TOKENS): read=True
        if any(t.lower() in low for t in WRITE_TOKENS): write=True
    if read and write:return 'READ_AND_WRITE_OR_CONTEXT_AMBIGUOUS'
    if read:return 'READ_LIKELY'
    if write:return 'WRITE_LIKELY'
    return 'REFERENCE_ONLY_OR_DYNAMIC'

def ref_matches(filename,ref):
    if ref.startswith('_'):
        return filename.lower().endswith(ref.lower())
    stem=Path(ref).stem
    ext=Path(ref).suffix
    return name_matches(filename,stem,ext)

asset_status=[]
for r in assets47:
    ref=r['literal_reference']
    if not r.get('looks_like_data_asset','').lower()=='true':
        continue
    matches=[]
    for root in ROOTS:
        for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
            dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
            for fn in files:
                if ref_matches(fn,ref):
                    matches.append(Path(here)/fn)
    cls=classify_ref(ref)
    asset_status.append({
      'literal_reference':ref,
      'runner_usage_classification':cls,
      'drive_match_count':len(matches),
      'drive_match_found':bool(matches),
      'example_matching_basename':matches[0].name if matches else '',
      'private_drive_path_exported':False
    })

likely_reads=[r for r in asset_status if r['runner_usage_classification'] in {'READ_LIKELY','READ_AND_WRITE_OR_CONTEXT_AMBIGUOUS'}]
missing_likely_reads=[r for r in likely_reads if not r['drive_match_found']]

print('Data-asset references audited:',len(asset_status))
print('Likely read dependencies:',len(likely_reads))
print('Likely read dependencies not found by basename/suffix search:',len(missing_likely_reads))
for r in missing_likely_reads[:20]:
    print(' MISSING?',r['literal_reference'],r['runner_usage_classification'])


Data-asset references audited: 20
Likely read dependencies: 7
Likely read dependencies not found by basename/suffix search: 0


In [6]:
# CODE CELL 6/9 — Determine whether frozen v15 exposes a generic event-construction API

function_names=[r['function_name'] for r in functions47 if r.get('function_name')]
builder_keywords=[
 'build_event','create_event','make_event','audit_event','transform_resource',
 'map_resource','process_resource','build_audit','create_audit','emit_event'
]
event_builder_functions=[
    name for name in function_names
    if any(k in name.lower() for k in builder_keywords)
]

# Conservative semantic inspection of function source names from the authoritative runner.
helper_only_expected={'nonempty_warning','compute_metrics','split_warning_tokens',
                      'join_warning_tokens','warning_count','sha256_file'}
helper_names=set(function_names)

generic_event_constructor_found=bool(event_builder_functions)

api_rows=[]
for name in sorted(set(function_names)):
    api_rows.append({
      'function_name':name,
      'appears_to_construct_or_transform_events':name in event_builder_functions,
      'classification':'EVENT_CONSTRUCTION_API' if name in event_builder_functions else 'METRIC_WARNING_OR_UTILITY_HELPER'
    })

print('Frozen runner functions:',sorted(set(function_names)))
print('Generic event-construction API found:',generic_event_constructor_found)
if not generic_event_constructor_found:
    print('The frozen runner exposes metric/warning utilities, not a generic new-resource transformation API.')


Frozen runner functions: ['compute_metrics', 'join_warning_tokens', 'nonempty_warning', 'sha256_file', 'split_warning_tokens', 'warning_count']
Generic event-construction API found: False
The frozen runner exposes metric/warning utilities, not a generic new-resource transformation API.


In [7]:
# CODE CELL 7/9 — Execute the bootstrap-only copy; do not patch failures

execution_status='NOT_RUN'
execution_seconds=None
execution_error=''
executed_sha=''
output_count_before=0
output_count_after=0

# Snapshot likely v15-named Drive artifacts before the replay.
def discover_v15_named():
    out=[]
    for root in ROOTS:
        for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
            dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
            for fn in files:
                low=fn.lower()
                if ('v15' in low or 'fidelity_audit' in low or 'medication_attribution' in low) and Path(fn).suffix.lower() in {'.csv','.json','.parquet','.zip'}:
                    out.append(Path(here)/fn)
    return out

before=discover_v15_named()
output_count_before=len(before)

try:
    import nbformat
    from nbclient import NotebookClient
    from nbclient.exceptions import CellExecutionError

    nb=nbformat.read(str(WRAPPER),as_version=4)
    t0=time.time()
    client=NotebookClient(
        nb,
        timeout=600,
        kernel_name='python3',
        resources={'metadata':{'path':str(ROOT)}},
        allow_errors=False
    )
    try:
        client.execute()
        execution_status='PASS_BOOTSTRAP_ONLY_V15_NATIVE_REPLAY'
    except CellExecutionError as e:
        execution_status='FAIL_FROZEN_METHOD_OR_ASSET_DEPENDENCY'
        execution_error=str(e)
    except Exception as e:
        execution_status='FAIL_EXECUTION_FRAMEWORK_OR_RUNTIME'
        execution_error=repr(e)
    execution_seconds=round(time.time()-t0,3)

    executed=WORK/'Phase48_executed_v15_wrapper.ipynb'
    nbformat.write(nb,str(executed))
    executed_sha=sha256(executed)
except Exception as e:
    execution_status='FAIL_EXECUTION_FRAMEWORK_OR_RUNTIME'
    execution_error=repr(e)

after=discover_v15_named()
output_count_after=len(after)

# Redact absolute Drive paths from public error text.
public_error=execution_error
public_error=re.sub(r'/content/drive/[^\s\'"]+','<DRIVE_PATH>',public_error)
public_error=re.sub(r'([A-Za-z]:\\\\[^\s\'"]+)','<LOCAL_PATH>',public_error)
public_error=public_error[-5000:]

print('Bootstrap-only frozen v15 replay:',execution_status)
print('Execution seconds:',execution_seconds)
print('v15-like Drive assets before/after:',output_count_before,output_count_after)
if public_error:
    print('Public-safe error tail:')
    print(public_error[-1500:])


/usr/local/lib/python3.13/dist-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


Bootstrap-only frozen v15 replay: PASS_BOOTSTRAP_ONLY_V15_NATIVE_REPLAY
Execution seconds: 40.372
v15-like Drive assets before/after: 52 52


In [8]:
# CODE CELL 8/9 — Arm C executability decision

# A native replay pass means only that v15 reproduces its own frozen workflow.
native_reproducible=(execution_status=='PASS_BOOTSTRAP_ONLY_V15_NATIVE_REPLAY')

# The actual neuroimaging Arm C requires an unmodified path from the study evidence to
# generic v15 audit events/rules. Metric-only helpers are insufficient.
if generic_event_constructor_found:
    armc_path_status='GENERIC_EVENT_API_FOUND_REQUIRES_DIRECT_NEUROIMAGING_EXECUTION'
    armc_reason='A possible frozen event-construction API exists; a separate direct study-data execution would still be required.'
else:
    armc_path_status='NO_GENERIC_NEW_RESOURCE_EVENT_API_IN_FROZEN_RUNNER'
    armc_reason=(
      'The authoritative frozen runner exposes warning/metric utilities but no generic event-construction '
      'or arbitrary-resource transformation API. Applying it to ImagingStudy/Device/Provenance would require '
      'new adapter semantics or rules, which would no longer be the unchanged v15 comparator.'
    )

armc_neuroimaging_executed=False
four_arm_evaluation=False

decision_rows=[
 {
  'criterion':'native_v15_own_workflow_reproducible',
  'passed':native_reproducible,
  'evidence':execution_status
 },
 {
  'criterion':'bootstrap_only_patch_preserved_method_cells',
  'passed':all(r['method_source_unchanged'] for r in patch_rows if int(r['cell_index'])!=mount_cells[0]),
  'evidence':f'only code cell {mount_cells[0]} had drive.mount bootstrap replacement'
 },
 {
  'criterion':'generic_event_construction_api_available',
  'passed':generic_event_constructor_found,
  'evidence':armc_path_status
 },
 {
  'criterion':'unmodified_neuroimaging_execution_path_available',
  'passed':False,
  'evidence':armc_reason
 },
 {
  'criterion':'frozen_generic_tfl_armC_executed_on_neuroimaging',
  'passed':False,
  'evidence':'No unchanged v15 path has processed the neuroimaging study evidence'
 }
]

print('Native v15 reproducible:',native_reproducible)
print('Arm C path status:',armc_path_status)
print('Neuroimaging Arm C executed:',armc_neuroimaging_executed)


Native v15 reproducible: True
Arm C path status: NO_GENERIC_NEW_RESOURCE_EVENT_API_IN_FROZEN_RUNNER
Neuroimaging Arm C executed: False


In [9]:
# CODE CELL 9/9 — Save public Phase 48 artifacts

def save(name,rows,fields=None):
    p=STAGE/name
    write_csv(p,rows,fields)
    return p

artifacts=[]
artifacts.append(save('ICHI2027_Phase48_BOOTSTRAP_PATCH_AUDIT.csv',patch_rows))
artifacts.append(save('ICHI2027_Phase48_EXTERNAL_ASSET_STATUS.csv',asset_status if asset_status else [
 {'literal_reference':'','runner_usage_classification':'NONE','drive_match_count':0,'drive_match_found':False,
  'example_matching_basename':'','private_drive_path_exported':False}
]))
artifacts.append(save('ICHI2027_Phase48_FROZEN_API_CLASSIFICATION.csv',api_rows if api_rows else [
 {'function_name':'','appears_to_construct_or_transform_events':False,'classification':'NO_FUNCTIONS'}
]))
artifacts.append(save('ICHI2027_Phase48_ARM_C_EXECUTABILITY_DECISION.csv',decision_rows))

exec_rows=[{
 'frozen_commit':COMMIT,
 'authoritative_runner_sha256':EXPECTED_RUNNER_SHA,
 'bootstrap_wrapper_sha256':sha256(WRAPPER),
 'mount_cell_index':mount_cells[0],
 'drive_mount_calls_replaced':patched_calls,
 'native_execution_status':execution_status,
 'execution_seconds':execution_seconds if execution_seconds is not None else '',
 'executed_notebook_sha256':executed_sha,
 'v15_named_assets_before':output_count_before,
 'v15_named_assets_after':output_count_after,
 'public_error_tail':public_error.replace('\n',' ')[:3000],
 'neuroimaging_armC_executed':False
}]
artifacts.append(save('ICHI2027_Phase48_NATIVE_REPLAY_AUDIT.csv',exec_rows))

readme=STAGE/'ICHI2027_Phase48_README.txt'
readme.write_text(
 'Phase 48 tests the frozen v15 runner using a bootstrap-only execution copy.\n'
 'Only the Drive mount statement may be replaced because the parent notebook already mounted Drive; all other frozen code cells are hash-checked as unchanged.\n'
 'A native replay pass establishes reproducibility of the original v15 workflow only.\n'
 'The frozen runner exposes metric/warning helpers but, unless explicitly discovered here, no generic arbitrary-resource event-construction API.\n'
 'No MRI-specific laterality, unit, longitudinal-pairing, model-version, provenance, or TRACE-state rule is added.\n'
 'Therefore native v15 reproducibility must not be mislabeled as neuroimaging Arm C execution.\n',
 encoding='utf-8'
)
artifacts.append(readme)

summary={
 'project':PROJECT,
 'phase':PHASE,
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'input_phase47_shareable_sha256':sha256(P['shareable']),
 'frozen_repository':REPO,
 'frozen_tag':TAG,
 'frozen_commit':COMMIT,
 'verified_archive_sha256':sha256(archive),
 'authoritative_runner_sha256':sha256(RUNNER),
 'bootstrap_wrapper_sha256':sha256(WRAPPER),
 'bootstrap_cells_modified':len(mount_cells),
 'drive_mount_calls_replaced':patched_calls,
 'non_bootstrap_method_cells_hash_preserved':all(
     r['method_source_unchanged'] for r in patch_rows if int(r['cell_index'])!=mount_cells[0]
 ),
 'external_data_asset_references':len(asset_status),
 'likely_read_dependencies':len(likely_reads),
 'missing_likely_read_dependencies':len(missing_likely_reads),
 'frozen_runner_function_names':sorted(set(function_names)),
 'generic_event_constructor_found':generic_event_constructor_found,
 'native_v15_execution_status':execution_status,
 'native_v15_reproducible_in_own_workflow':native_reproducible,
 'armC_execution_path_status':armc_path_status,
 'frozen_original_generic_tfl_armC_executed':False,
 'armC_neuroimaging_executed':False,
 'armA_evidence_supported_gates_carried_forward':p47['armA_evidence_supported_gates_carried_forward'],
 'armA_remaining_gates_carried_forward':p47['armA_remaining_gates_carried_forward'],
 'official_micdm_load_executed':False,
 'official_omop_etl_executed':False,
 'real_authorized_mri_processed':False,
 'full_A_B_C_D_evaluation':False,
 'artifact_sha256':{p.name:sha256(p) for p in artifacts},
 'warning':'A bootstrap-only native v15 replay does not establish neuroimaging Arm C. The unchanged frozen runner must expose and execute a compatible study-data path; otherwise v15 remains prior-art/reference evidence rather than an executed quantitative arm.'
}

summary_path=STAGE/'ICHI2027_Phase48_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

zip_path=STAGE/'ICHI2027_Phase48_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[zip_path]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2: raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase 48 completed.')
print('Native v15:',execution_status)
print('Arm C path:',armc_path_status)
print('Neuroimaging Arm C executed: False')
print('Shareable:',OUT/summary_path.name)
print('Public ZIP:',OUT/zip_path.name)


SUCCESS: Phase 48 completed.
Native v15: PASS_BOOTSTRAP_ONLY_V15_NATIVE_REPLAY
Arm C path: NO_GENERIC_NEW_RESOURCE_EVENT_API_IN_FROZEN_RUNNER
Neuroimaging Arm C executed: False
Shareable: /content/drive/MyDrive/ICHI2027/Phase48/run_20261001T035922_093456Z/ICHI2027_Phase48_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase48/run_20261001T035922_093456Z/ICHI2027_Phase48_PUBLIC_OUTPUTS.zip


## Decision after Phase 48

If the native replay succeeds but no generic new-resource event API exists, additional attempts to force the frozen runner onto MRI would weaken the comparison by modifying the comparator.

At that point the study should preserve v15 as **verified prior-art / generic-method evidence** and explicitly amend the quantitative experiment rather than pretending the original four-arm plan was executable unchanged.

If a genuine frozen generic event API is discovered, the next phase can apply that API directly to the preregistered E0–E7 evidence without introducing neuroimaging-specific rules.
